# Session 16: LLM Fine-Tuning and AI Agents

**Learning goals:**

- Explain the roles of pre-training, post-training and supervised fine-tuning.
- Train a LoRA adapter, evaluate it on held-out tickets and explain what QLoRA quantises.
- Trace an agent tool call and decide when a fixed workflow is sufficient.

**Prerequisites:**

- Next-token prediction, ticket classification and model evaluation.
- Python functions and dictionaries, and basic PyTorch training.
- Internet access and a Colab T4 GPU for the fine-tuning practice.

## Outline

1. From classification to generation
2. LLM: Pre-training and Post-training
3. Fine-tune a small language model
4. Introduction to AI agents
5. Design a support ticket agent
6. Review and compare

## 1. From Classification to Generation

A classifier chooses a label from a fixed set. A generative LLM scores possible next tokens, and the generation process appends the selected token ID and repeats.

Tokenisation splits text into tokens and maps each token to its vocabulary ID. The embedding layer looks up a vector for each ID. The diagram uses GPT-2 IDs and hides leading spaces in the token labels.

![The tokeniser maps the Wi-Fi ticket to vocabulary IDs. The embedding layer looks up vectors from those IDs. The selected token ID is appended to the input sequence for the next generation step.](images/llm-generation-image2.png)

## 2. LLM: Pre-training and Post-training

### Pre-training

During pre-training, the model learns to predict the next token in text. For example, given **My account is**, it learns to predict **locked** from the original text. The result is a **base model**, such as the original GPT-2 checkpoint.

### Post-training

One way to post-train a model is **supervised fine-tuning (SFT)**. The model learns from an instruction and its expected answer. For example, it learns to answer **Account access** when asked to classify a locked-account ticket.

**LoRA** keeps the existing model weights fixed and trains a small adapter. The adapter learns two small matrices whose product gives a low-rank weight update. The lab applies LoRA to an existing instruct model.

To reduce memory use, **QLoRA** stores the frozen linear-layer weights using a 4-bit encoding such as NF4 while training the LoRA adapter.

**Reinforcement learning (RL)** is also common in modern LLM post-training. When applied after SFT, it uses rewards for the model’s own responses to reinforce behaviours such as answering correctly and following instructions.

![Pre-training learns to predict the next token in text. In supervised fine-tuning with LoRA, the model learns from an instruction and target answer while training only the adapter.](images/pre-training-and-post-training-image2.png)


## 3. Fine-Tune a Small Language Model

[Download the fine-tuning practice notebook](<practice/Session16 Fine-Tune Support Ticket LLM Practice.ipynb>){download="Session16 Fine-Tune Support Ticket LLM Practice.ipynb"} · [Open in Colab](https://colab.research.google.com/github/wlos/AI_SS/blob/main/16-LLM-Fine-Tuning-and-Agents/practice/Session16%20Fine-Tune%20Support%20Ticket%20LLM%20Practice.ipynb)

The lab starts from the [Qwen2.5-0.5B-Instruct checkpoint](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct).

Use the [fictional support tickets](sample_data/support_tickets.csv) from Sessions 14–15. The practice keeps the original training and test splits.

1. Measure the original instruct model on the held-out tickets.
2. Choose LoRA or QLoRA on a Colab T4. Use Hugging Face PEFT to add an adapter, inspect the trainable parameter count and run one training pass with Transformers Trainer.
3. Evaluate the adapted model with the same tickets, prompt and generation settings. Calculate the proportion of replies that match the JSON schema and the proportion that give the correct category. Count an invalid reply as a classification failure.
4. Inspect two errors. Decide whether each is a formatting error, a wrong category or an ambiguous ticket.
5. Export the trained adapter as `session16-ticket-adapter.zip` for the agent practice.

If a GPU is unavailable, use the notebook’s quick mode to try a small subset on CPU.

## 4. Introduction to AI Agents

An **AI agent** is a system that chooses actions to work towards a goal. In an LLM agent, the language model decides what to do next and the application carries out permitted actions.

![A request goes to an LLM, which produces a response and can exchange information with retrieval, tools and memory. The LLM can also delegate a task to another agent when that capability is available.](images/ai-agent-architecture-image2.png)

Adapted from Anthropic’s augmented LLM diagram, with optional delegation to another agent. [Building effective agents](https://www.anthropic.com/research/building-effective-agents)

### Example applications

| Application | What the agent does | Example |
|---|---|---|
| Software development | Examines a repository, plans a change and edits code on a branch for pull request review. | [GitHub Copilot cloud agent](https://docs.github.com/en/copilot/how-tos/copilot-on-github/use-copilot-agents/research-plan-iterate) |
| Research | Makes a research plan, searches sources and produces a report. | [Google Gemini Deep Research](https://gemini.google/overview/deep-research/) |
| Customer service | Uses a knowledge base to handle common requests and hands complex cases to a person. | [Salesforce Agentforce Service Agent](https://trailhead.salesforce.com/content/learn/modules/agentforce-service-agent-quick-look/get-started-with-agentforce-service-agent) |

### How do LLM agents work?

An agent takes an **action** in its environment and uses the **feedback** to decide what to do next. The environment can be a system the agent interacts with, such as a support portal or a code repository.

![The human and agent can exchange messages. An action goes from the agent to the environment, and feedback comes back to the agent. The agent can take another action or stop.](images/ai-agent-action-feedback-image2.png)

Adapted from [Anthropic’s agent diagram](https://www.anthropic.com/research/building-effective-agents#agents).

1. **Read the request.** Use the goal, instructions and information already available.
2. **Choose an action.** The LLM can request a tool call or ask the user for more information.
3. **Get feedback.** The application checks and runs the requested tool, then returns its result to the LLM.
4. **Decide whether to continue.** The LLM can choose another action or give a final answer. The application stops when the task is complete or a limit is reached.

## 5. Design a Support Ticket Agent

Both designs use Qwen to decide whether to ask a follow-up question and to draft a reply. The classification tool supplies the category. The application uses that category to query the [fictional support guides](sample_data/support_guides.json). The sequence diagrams show a ticket with enough detail to classify.

![A student reports a locked account. The LLM requests an account access guide. The application runs the lookup and returns the guide result. The LLM then directs the student to the password reset page.](images/ai-agent-tool-loop-image2.png)

### Use the Session 16 adapter

Upload `session16-ticket-adapter.zip` from Section 3. The practice loads Qwen once. With the adapter enabled, it classifies the ticket. With the adapter disabled, it decides whether to ask for more information and writes the final reply.

![The app asks one Qwen model to decide with the adapter off, classify with the adapter on, and draft a reply with the adapter off after the guide lookup.](images/ticket-agent-adapter-sequence-image2.png)

[Download the adapter agent practice notebook](<practice/Session16 Ticket Agent Practice - LoRA Adapter.ipynb>){download="Session16 Ticket Agent Practice - LoRA Adapter.ipynb"} · [Open in Colab](https://colab.research.google.com/github/wlos/AI_SS/blob/main/16-LLM-Fine-Tuning-and-Agents/practice/Session16%20Ticket%20Agent%20Practice%20-%20LoRA%20Adapter.ipynb)

### Use a saved classifier from Sessions 14–15

Load a saved DistilBERT, GPT-2 or word-count MLP classifier as a separate local tool. Qwen handles the conversation, and the classifier returns a category for the same guide lookup.

![Qwen makes the decision and drafts the reply. A separate saved classifier returns the category, which the app uses to find the matching JSON guide.](images/ticket-agent-saved-classifier-sequence-image2.png)

[Download the saved-classifier agent practice notebook](<practice/Session16 Ticket Agent Practice - Saved Classifier.ipynb>){download="Session16 Ticket Agent Practice - Saved Classifier.ipynb"} · [Open in Colab](https://colab.research.google.com/github/wlos/AI_SS/blob/main/16-LLM-Fine-Tuning-and-Agents/practice/Session16%20Ticket%20Agent%20Practice%20-%20Saved%20Classifier.ipynb)

### Try the ticket chat

After classification, both designs use the guide result to answer the user.

| Ticket | Expected behaviour |
|---|---|
| “I can’t log in.” | Ask which service is affected. |
| “My student portal account is locked. What should I do?” | Look up the account access guide and use the result to answer. |
| “My laptop battery has stopped charging. What should I do?” | Look up the hardware guide and use the result to answer. |

Start either chat. Try the first message and give the second as a follow-up if the agent asks for details. Then try the hardware ticket as a new request. Type `/exit` and inspect the last trace to see the decision, category and guide result.